In [46]:
# Imports and Setup
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif, SelectKBest
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import pickle
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [47]:
# Load and Prepare Data
data = pd.read_csv('../data/updated_ckd_dataset_with_stages.csv')
target_col = 'ckd_pred'

columns_to_drop = ['ckd_stage', 'cluster']
for col in columns_to_drop:
    if col in data.columns:
        data = data.drop(col, axis=1)

print("Original class distribution:")
print(data[target_col].value_counts().sort_index())


Original class distribution:
ckd_pred
CKD       3875
No CKD     125
Name: count, dtype: int64


In [48]:
#Binary Classification Conversion
def convert_to_binary(y):
    """Convert to binary classification intelligently"""
    unique_classes = y.unique()
    
    if len(unique_classes) == 2:
        y_binary = (y == unique_classes[1]).astype(int)
        return y_binary
    
    if y.dtype == 'object':
        healthy_labels = ['Non-CKD', 'no', 'NO', 'Non CKD', 'non-ckd', 'notckd']
        healthy_label = None
        for label in unique_classes:
            if str(label).strip() in healthy_labels:
                healthy_label = label
                break
        
        if healthy_label is not None:
            y_binary = (y != healthy_label).astype(int)
        else:
            y_binary = (y != unique_classes[0]).astype(int)
        return y_binary
    
    y_binary = (y != min(unique_classes)).astype(int)
    return y_binary

y_original = data[target_col]
y_binary = convert_to_binary(y_original)

print("\nBinary distribution:")
print(pd.Series(y_binary).value_counts().sort_index())

if len(np.unique(y_binary)) < 2:
    raise ValueError("Only one class after conversion!")

data['Target'] = y_binary
data = data.drop(target_col, axis=1)


Binary distribution:
ckd_pred
0    3875
1     125
Name: count, dtype: int64


In [49]:
# Save Label Encoder
import pickle
from sklearn.preprocessing import LabelEncoder

# Create label encoder for the binary classes
label_encoder = LabelEncoder()
label_encoder.fit(['Healthy', 'CKD'])  # 0 = Healthy, 1 = CKD

# Save it
import os
os.makedirs('../models', exist_ok=True)

with open('../models/label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

print("Saved label encoder")

Saved label encoder


In [50]:
# Feature Selection
x = data.drop('Target', axis=1)
y = data['Target']

num_cols = x.select_dtypes(include=np.number).columns.tolist()
cat_cols = x.select_dtypes(include='object').columns.tolist()

x_imputed = x.copy()

if len(num_cols) > 0:
    imputer_num = SimpleImputer(strategy='median')
    x_imputed[num_cols] = imputer_num.fit_transform(x[num_cols])

if len(cat_cols) > 0:
    imputer_cat = SimpleImputer(strategy='most_frequent')
    x_imputed[cat_cols] = imputer_cat.fit_transform(x[cat_cols])
    
    for col in cat_cols:
        le = LabelEncoder()
        x_imputed[col] = le.fit_transform(x_imputed[col].astype(str))

k_features = min(15, x_imputed.shape[1])
selector = SelectKBest(mutual_info_classif, k=k_features)
x_selected = selector.fit_transform(x_imputed, y)

selected_features = x.columns[selector.get_support()].tolist()
print(f"\nSelected {len(selected_features)} features:")
for i, feat in enumerate(selected_features, 1):
    print(f"{i:2d}. {feat}")

x_reduced = x[selected_features].copy()


Selected 15 features:
 1. serum_creatinine
 2. gfr
 3. bun
 4. serum_calcium
 5. ana
 6. c3_c4
 7. hematuria
 8. oxalate_levels
 9. urine_ph
10. blood_pressure
11. diet
12. water_intake
13. painkiller_usage
14. family_history
15. weight_changes


In [51]:
# Train/Test Split
min_class_count = min(Counter(y).values())

if min_class_count < 2:
    x_train, x_test, y_train, y_test = train_test_split(
        x_reduced, y, test_size=0.2, random_state=RANDOM_STATE
    )
else:
    x_train, x_test, y_train, y_test = train_test_split(
        x_reduced, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

print(f"\nTrain: {len(y_train)} samples - {dict(Counter(y_train))}")
print(f"Test: {len(y_test)} samples - {dict(Counter(y_test))}")

if len(np.unique(y_test)) < 2:
    raise ValueError("Test set must contain both classes!")



Train: 3200 samples - {0: 3100, 1: 100}
Test: 800 samples - {0: 775, 1: 25}


In [52]:
# SMOTE Preprocessing Function
def preprocess_for_smote(x_c):
    """Impute, encode categoricals, and standardize features"""
    x_c = x_c.copy()
    
    num_cols = x_c.select_dtypes(include=np.number).columns.tolist()
    cat_cols = x_c.select_dtypes(include='object').columns.tolist()
    
    if len(num_cols) > 0:
        imputer_num = SimpleImputer(strategy='median')
        x_c[num_cols] = imputer_num.fit_transform(x_c[num_cols])
    
    if len(cat_cols) > 0:
        imputer_cat = SimpleImputer(strategy='most_frequent')
        x_c[cat_cols] = imputer_cat.fit_transform(x_c[cat_cols])
        
        for col in cat_cols:
            le = LabelEncoder()
            x_c[col] = le.fit_transform(x_c[col].astype(str))
    
    x_c_array = x_c.values
    scaler = StandardScaler()
    x_c_scaled = scaler.fit_transform(x_c_array)
    
    return x_c_scaled

In [53]:
# Apply SMOTE 
x_train_processed = preprocess_for_smote(x_train)

smote = SMOTE(
    sampling_strategy=0.6,
    random_state=RANDOM_STATE, 
    k_neighbors=min(5, min(Counter(y_train).values()) - 1)
)
x_train_balanced, y_train_balanced = smote.fit_resample(x_train_processed, y_train)

print(f"\nSMOTE Results:")
print(f"Before: {dict(Counter(y_train))}")
print(f"After: {dict(Counter(y_train_balanced))}")


SMOTE Results:
Before: {0: 3100, 1: 100}
After: {0: 3100, 1: 1860}


In [ ]:
#Create Stratified Clients
num_clients = 4
train_data_balanced = pd.DataFrame(x_train_balanced)
train_data_balanced['Target'] = y_train_balanced

min_train_class = min(Counter(y_train_balanced).values())
if min_train_class < num_clients:
    num_clients = min_train_class

skf = StratifiedKFold(n_splits=num_clients, shuffle=True, random_state=RANDOM_STATE)

client_data_processed = []
print(f"\nCreating {num_clients} clients:")
for fold_idx, (_, client_idx) in enumerate(skf.split(
    train_data_balanced.drop('Target', axis=1),
    train_data_balanced['Target']
)):
    client_subset = train_data_balanced.iloc[client_idx]
    x_c = client_subset.drop('Target', axis=1).values
    y_c = client_subset['Target'].values
    
    client_data_processed.append((x_c, y_c))
    print(f"Client {fold_idx+1}: {len(y_c)} samples - {dict(Counter(y_c))}")


In [55]:
# Compute Class Weights
all_labels = np.concatenate([y_c for _, y_c in client_data_processed])
unique_classes = np.unique(all_labels)
class_weights_array = compute_class_weight(
    'balanced', classes=unique_classes, y=all_labels
)

class_weights = {i: np.power(weight, 0.25) for i, weight in enumerate(class_weights_array)}

print("\nClass weights:")
for cls, weight in class_weights.items():
    class_name = "Healthy" if cls == 0 else "CKD"
    print(f"{class_name} ({cls}): {weight:.4f}")


Class weights:
Healthy (0): 0.9457
CKD (1): 1.0746


In [56]:
# Process Test Data and Save
x_test_processed = preprocess_for_smote(x_test)

os.makedirs('../data', exist_ok=True)

with open('../data/client_data_processed.pkl', 'wb') as f:
    pickle.dump(client_data_processed, f)

with open('../data/class_weights.pkl', 'wb') as f:
    pickle.dump(class_weights, f)

with open('../data/test_set.pkl', 'wb') as f:
    pickle.dump((x_test_processed, y_test.values), f)

with open('../data/selected_features.pkl', 'wb') as f:
    pickle.dump(selected_features, f)

print("\nFiles saved successfully")
print(f"Features: {x.shape[1]} → {len(selected_features)}")
print(f"Training samples: {len(y_train)} → {len(y_train_balanced)}")
print(f"Test samples: {len(y_test)}")


Files saved successfully
Features: 20 → 15
Training samples: 3200 → 4960
Test samples: 800
